# **Babysitting ResNet50 (from scratch) - Loss**

1. **Cella 1 — Import Centralizzati, Configurazione, Riproducibilità e Ispezione Grezza (Sezioni 1–5)**

In [ ]:
# ==============================================================================
# SEZIONE 1: IMPORT CENTRALIZZATI DELLE LIBRERIE
# ==============================================================================
import os
import gc
import copy
import json
import time
import random
import shutil
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torchvision.models.segmentation import fcn_resnet50

from pycocotools.coco import COCO


# ==============================================================================
# SEZIONE 2: CONFIGURAZIONE CENTRALIZZATA, CARTELLE OUTPUT E FLAG DI CONTROLLO
# ==============================================================================
SEED = 42
CROP_SIZE = 256            # Risoluzione spaziale unica (intero) validata nello Step 3-BIS
DEFAULT_BATCH_SIZE = 8     # Valore iniziale di fallback prima della calibrazione GPU
IGNORE_INDEX = 255         # Etichetta standard per pixel da escludere da loss e metriche
NUM_CLASSES = 81           # 0 = background + 80 classi COCO Thing
AUX_LOSS_WEIGHT = 0.5      # Peso del ramo ausiliario FCN (standard torchvision)

# Statistiche di normalizzazione ImageNet (standard per backbone ResNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Parametri operativi della pipeline dati
NUM_WORKERS = 2             # Ottimale per ambienti Kaggle (evita contesa thread CPU)
TRAIN_MAX_SAMPLES = 16   # None = intero dataset (118.287 img), oppure int (es. 12000) per subset bilanciato
CACHE_IN_RAM = True         # Imposta True SOLO se TRAIN_MAX_SAMPLES <= 15000

# Flag di controllo esecuzione (imposta False dopo aver già salvato grafici e check diagnostici)
RUN_RAW_INSPECTION = True  # Ispezione directory ed estensioni COCO (< 1s)
RUN_EDA_PLOTS = True       # Se True, rigenera e salva i grafici di uniformità e Dumbbell Chart
RUN_BABYSITTING = True     # Se True, esegue i test diagnostici Step 3 (80 ep) e Step 3-BIS (~12 min)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cartelle persistenti per esportazione figure (PNG/SVG) e salvataggio pesi (Step 4+)
BASE_WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
FIGURES_DIR = os.path.join(BASE_WORKING_DIR, "figures")
CHECKPOINTS_DIR = os.path.join(BASE_WORKING_DIR, "checkpoints")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)


def save_and_show_fig(
    fig: plt.Figure,
    filename: str,
    dpi: int = 300,
    save_svg: bool = False,
) -> str:
    """
    Salva la figura in PNG ad alta risoluzione con sfondo bianco opaco
    e, se richiesto (grafici analitici senza imshow), esporta anche la versione vettoriale SVG.
    """
    out_path = os.path.join(FIGURES_DIR, filename)
    fig.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    print(f"[Figura PNG salvata] -> {out_path}")

    if save_svg:
        svg_name = os.path.splitext(filename)[0] + ".svg"
        svg_path = os.path.join(FIGURES_DIR, svg_name)
        fig.savefig(svg_path, bbox_inches="tight", facecolor="white")
        print(f"[Figura SVG salvata] -> {svg_path}")

    plt.show()
    plt.close(fig)
    return out_path


def denormalize_image(tensor_img: torch.Tensor) -> np.ndarray:
    """Converte un tensore immagine normalizzato ImageNet (3, H, W) in array RGB [0, 1]."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img = tensor_img.cpu() * std + mean
    return torch.clamp(img, 0.0, 1.0).permute(1, 2, 0).numpy()


def find_coco_root(base_input_dir: str = "/kaggle/input") -> str:
    """Individua automaticamente la directory radice di COCO 2017 dentro /kaggle/input."""
    if not os.path.exists(base_input_dir):
        raise FileNotFoundError(f"Directory base '{base_input_dir}' non trovata.")

    for root, dirs, _ in os.walk(base_input_dir):
        if {"train2017", "val2017", "annotations"}.issubset(set(dirs)):
            return root

    raise FileNotFoundError(
        f"Impossibile trovare una directory COCO 2017 valida dentro '{base_input_dir}'. "
        "Verifica i dataset montati nel pannello Input di Kaggle."
    )


print("=== CONFIGURAZIONE AMBIENTE ===")
print(f"PyTorch version     : {torch.__version__}")
print(f"Torchvision version : {torchvision.__version__}")
print(f"Device selezionato  : {DEVICE}")
print(f"CROP_SIZE fissato   : {CROP_SIZE}x{CROP_SIZE} px | AUX_LOSS_WEIGHT: {AUX_LOSS_WEIGHT}")
print(f"Cartella Figure     : {FIGURES_DIR}")
print(f"Cartella Checkpoints: {CHECKPOINTS_DIR}")

if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"Numero GPU rilevate : {num_gpus}")
    for gpu_idx in range(num_gpus):
        props = torch.cuda.get_device_properties(gpu_idx)
        vram_gb = props.total_memory / (1024 ** 3)
        print(f"  - GPU [{gpu_idx}]: {props.name} ({vram_gb:.2f} GB VRAM)")
else:
    print("ATTENZIONE: Nessuna GPU CUDA rilevata. Attiva l'acceleratore GPU nelle impostazioni Kaggle.")

DATA_ROOT = find_coco_root("/kaggle/input")
print(f"DATA_ROOT rilevato  : {DATA_ROOT}")


# ==============================================================================
# SEZIONE 3: RIPRODUCIBILITÀ DETERMINISTICA E PALETTE COLORI
# ==============================================================================
def seed_everything(seed: int = SEED) -> None:
    """Blocca i generatori pseudo-casuali su Python, NumPy e PyTorch."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int) -> None:
    """Garantisce seed deterministici per ogni processo worker del DataLoader."""
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything(SEED)

# Palette deterministica di 81 colori (0 = nero per background)
_rng_palette = np.random.default_rng(SEED)
PALETTE = np.zeros((NUM_CLASSES, 3), dtype=np.uint8)
PALETTE[1:] = _rng_palette.integers(40, 255, size=(NUM_CLASSES - 1, 3), dtype=np.uint8)
print(f"Seed deterministico fissato a: {SEED}")


# ==============================================================================
# SEZIONE 4 & 5: ISPEZIONE STRUTTURA DIRECTORY E DATI GREZZI
# ==============================================================================
def inspect_raw_coco_directory(data_root: str) -> None:
    """Verifica struttura cartelle, estensioni file, metadati JSON e statistiche immagini."""
    print("\n=== ISPEZIONE STRUTTURA DIRECTORY COCO 2017 ===")
    root_entries = sorted(os.listdir(data_root))
    print(f"Contenuto di DATA_ROOT: {root_entries}")

    possible_mask_dirs = [
        d for d in root_entries
        if os.path.isdir(os.path.join(data_root, d))
        and ("label" in d.lower() or "mask" in d.lower() or "stuff" in d.lower())
    ]
    print(f"Cartelle maschere semantiche pre-renderizzate: {possible_mask_dirs if possible_mask_dirs else 'NESSUNA'}")

    annotations_dir = os.path.join(data_root, "annotations")
    if os.path.exists(annotations_dir):
        ann_files = sorted(os.listdir(annotations_dir))
        print(f"File in annotations/ ({len(ann_files)}):")
        for f_name in ann_files:
            f_size_mb = os.path.getsize(os.path.join(annotations_dir, f_name)) / (1024 ** 2)
            print(f"  - {f_name} ({f_size_mb:.2f} MB)")

    split_files = {}
    for split in ["train2017", "val2017", "test2017"]:
        split_path = os.path.join(data_root, split)
        if os.path.exists(split_path):
            files = sorted(os.listdir(split_path))
            split_files[split] = files
            ext_counts = Counter(os.path.splitext(f)[1].lower() for f in files)
            print(f"Split '{split}': {len(files)} file | Estensioni: {dict(ext_counts)}")
        else:
            split_files[split] = []

    if len(split_files["val2017"]) > 0:
        inspect_limit = min(500, len(split_files["val2017"]))
        modes_counter = Counter()
        shapes_set = set()
        heights, widths = [], []

        for fname in split_files["val2017"][:inspect_limit]:
            with Image.open(os.path.join(data_root, "val2017", fname)) as im:
                modes_counter[im.mode] += 1
                w, h = im.size
                widths.append(w)
                heights.append(h)
                shapes_set.add((h, w))

        print(f"\nStatistiche su {inspect_limit} immagini di 'val2017':")
        print(f"  - Modalità colore (PIL mode) : {dict(modes_counter)}")
        print(f"  - Risoluzioni distinte       : {len(shapes_set)} su {inspect_limit} immagini")
        print(f"  - Altezza H (min / med / max): {min(heights)} / {int(np.median(heights))} / {max(heights)}")
        print(f"  - Larghezza W (min/ med/ max): {min(widths)} / {int(np.median(widths))} / {max(widths)}")


if RUN_RAW_INSPECTION:
    inspect_raw_coco_directory(DATA_ROOT)

2. **Cella 2 — Caricamento Annotazioni JSON (pycocotools) e Mappatura Tassonomica (Sezione 6 - Parte 1)**

In [ ]:
# ==============================================================================
# SEZIONE 6 (PARTE 1): CARICAMENTO ISTANZE COCO JSON E MAPPATURA CLASSI
# (Si esegue una sola volta all'avvio della sessione)
# ==============================================================================
t0 = time.perf_counter()
train_ann_path = os.path.join(DATA_ROOT, "annotations", "instances_train2017.json")
val_ann_path = os.path.join(DATA_ROOT, "annotations", "instances_val2017.json")

coco_train = COCO(train_ann_path)
coco_val = COCO(val_ann_path)
print(f"Annotazioni JSON caricate in {time.perf_counter() - t0:.2f} secondi.")

# Mappatura biunivoca: COCO category_id (1..90) <-> indice contiguo (1..80), 0 = background
sorted_cat_ids = sorted(coco_train.getCatIds())
COCO_ID_TO_CONTIGUOUS = {cat_id: idx + 1 for idx, cat_id in enumerate(sorted_cat_ids)}
CONTIGUOUS_TO_NAME = {0: "background"}
for cat_info in coco_train.loadCats(sorted_cat_ids):
    CONTIGUOUS_TO_NAME[COCO_ID_TO_CONTIGUOUS[cat_info["id"]]] = cat_info["name"]

print(f"Mappatura completata: {len(COCO_ID_TO_CONTIGUOUS)} classi Thing -> indici [1..80], 0 = background.")

3. **Cella 3 — Definizione COCOSemanticDataset, Campionamento Bilanciato e Analisi EDA (Sezione 6 - Parte 2 & Sezione 7)**

In [ ]:
# ==============================================================================
# SEZIONE 6 (PARTE 2) & SEZIONE 7: DEFINIZIONE DATASET, CAMPIONAMENTO E ANALISI EDA
# (Nessun caricamento pesante: modificabile istantaneamente senza ricaricare i JSON)
# ==============================================================================
def analyze_coco_split(coco_api: COCO, split_name: str, target_short_side: int = CROP_SIZE) -> Counter:
    """Analizza statisticamente lo split JSON e misura l'impatto del resize su oggetti piccoli."""
    img_ids = coco_api.getImgIds()
    imgs_info = coco_api.loadImgs(img_ids)

    small_side_count = 0
    empty_imgs_count = 0
    class_img_counts = Counter()
    orig_rel_areas = []
    resized_min_dims = []
    tiny_after_resize = 0  # Oggetti con lato equivalente < 8 px post-resize (output stride FCN)

    for info in imgs_info:
        h, w = info["height"], info["width"]
        short_side = min(h, w)
        if short_side < target_short_side:
            small_side_count += 1

        scale = target_short_side / float(short_side)
        anns = coco_api.loadAnns(coco_api.getAnnIds(imgIds=info["id"]))

        if len(anns) == 0:
            empty_imgs_count += 1
            continue

        present_classes = set()
        for ann in anns:
            c_idx = COCO_ID_TO_CONTIGUOUS[ann["category_id"]]
            present_classes.add(c_idx)

            area = ann["area"]
            orig_rel_areas.append(area / float(h * w))
            equiv_side = np.sqrt(area * (scale ** 2))
            resized_min_dims.append(equiv_side)
            if equiv_side < 8.0:
                tiny_after_resize += 1

        for c_idx in present_classes:
            class_img_counts[c_idx] += 1

    sorted_by_img = sorted(class_img_counts.items(), key=lambda x: x[1])
    rarest_5 = [(CONTIGUOUS_TO_NAME[k], v) for k, v in sorted_by_img[:5]]
    common_5 = [(CONTIGUOUS_TO_NAME[k], v) for k, v in sorted_by_img[-5:][::-1]]
    imbalance_ratio = sorted_by_img[-1][1] / max(1, sorted_by_img[0][1])

    print(f"\n=== ANALISI STATISTICA COMPLETA: {split_name} ({len(img_ids)} immagini) ===")
    print(f"  - Immagini senza oggetti (solo background) : {empty_imgs_count} ({100*empty_imgs_count/len(img_ids):.2f}%)")
    print(f"  - Immagini con lato corto < {target_short_side} px         : {small_side_count} ({100*small_side_count/len(img_ids):.2f}%)")
    print(f"  - Classi coperte                           : {len(class_img_counts)} / 80")
    print(f"  - Top 5 classi più frequenti (n. immagini) : {common_5}")
    print(f"  - Top 5 classi più rare (n. immagini)      : {rarest_5}")
    print(f"  - Rapporto squilibrio (max / min immagini) : {imbalance_ratio:.1f}x")
    print(f"  - Area relativa media singolo oggetto      : {100*np.mean(orig_rel_areas):.2f}% dell'immagine")
    print(f"  - Istanze con lato equiv. < 8px post-resize: {tiny_after_resize} / {len(resized_min_dims)} "
          f"({100*tiny_after_resize/max(1, len(resized_min_dims)):.2f}%)")

    return class_img_counts


def select_class_balanced_subset(coco_api: COCO, max_samples: int, seed: int = SEED) -> list:
    """
    Seleziona un sottoinsieme di 'max_samples' image_id massimizzando la copertura
    delle classi rare tramite campionamento greedy per quota minima.
    """
    rng = random.Random(seed)
    all_img_ids = sorted(coco_api.getImgIds())
    if max_samples is None or max_samples >= len(all_img_ids):
        return all_img_ids

    cat_to_imgs = defaultdict(list)
    img_to_cats = {}

    for img_id in all_img_ids:
        anns = coco_api.loadAnns(coco_api.getAnnIds(imgIds=img_id))
        cats = sorted({COCO_ID_TO_CONTIGUOUS[a["category_id"]] for a in anns})
        img_to_cats[img_id] = cats
        for c in cats:
            cat_to_imgs[c].append(img_id)

    for c in cat_to_imgs:
        rng.shuffle(cat_to_imgs[c])

    rare_first_classes = sorted(cat_to_imgs.keys(), key=lambda c: len(cat_to_imgs[c]))
    min_quota_per_class = max(1, max_samples // len(rare_first_classes))

    selected_set = set()
    selected_class_counts = Counter()

    for c in rare_first_classes:
        for img_id in cat_to_imgs[c]:
            if selected_class_counts[c] >= min_quota_per_class or len(selected_set) >= max_samples:
                break
            if img_id not in selected_set:
                selected_set.add(img_id)
                for present_c in img_to_cats[img_id]:
                    selected_class_counts[present_c] += 1

    non_empty_pool = [i for i in all_img_ids if len(img_to_cats[i]) > 0 and i not in selected_set]
    rng.shuffle(non_empty_pool)
    for img_id in non_empty_pool:
        if len(selected_set) >= max_samples:
            break
        selected_set.add(img_id)
        for present_c in img_to_cats[img_id]:
            selected_class_counts[present_c] += 1

    selected_list = sorted(selected_set)
    min_c = min(selected_class_counts.values()) if selected_class_counts else 0
    max_c = max(selected_class_counts.values()) if selected_class_counts else 0
    print(f"[Subset Bilanciato] Selezionate {len(selected_list)} immagini | "
          f"Classi presenti: {len(selected_class_counts)}/80 | "
          f"Min/Max img per classe: {min_c} / {max_c}")
    return selected_list


def compute_class_counts_from_ids(coco_api: COCO, img_ids: list) -> Counter:
    """Conta in quante immagini compare ciascuna delle 80 classi nel sottoinsieme dato."""
    counts = Counter()
    for img_id in img_ids:
        anns = coco_api.loadAnns(coco_api.getAnnIds(imgIds=img_id))
        present_classes = {COCO_ID_TO_CONTIGUOUS[a["category_id"]] for a in anns}
        for c_idx in present_classes:
            counts[c_idx] += 1
    return counts


def compute_lorenz_and_gini(sorted_asc_values: np.ndarray):
    """Calcola coordinate (x, y) della Curva di Lorenz e Coefficiente di Gini."""
    n = len(sorted_asc_values)
    total = np.sum(sorted_asc_values)
    cum_x = np.linspace(0.0, 1.0, n + 1)
    cum_y = np.concatenate([[0.0], np.cumsum(sorted_asc_values) / total])
    area_under_lorenz = np.trapezoid(cum_y, cum_x) if hasattr(np, "trapezoid") else np.trapz(cum_y, cum_x)
    gini = 1.0 - 2.0 * area_under_lorenz
    return cum_x, cum_y, gini


def plot_dataset_uniformity(
    train_counts: Counter,
    val_counts: Counter,
    balanced_subset_counts: Counter = None,
    subset_size: int = None,
) -> None:
    """Genera e salva il grafico di distribuzione logaritmica + Curva di Lorenz / Indice di Gini."""
    all_c_indices = list(range(1, NUM_CLASSES))
    sorted_desc_indices = sorted(all_c_indices, key=lambda c: train_counts[c], reverse=True)
    train_vals_desc = np.array([train_counts[c] for c in sorted_desc_indices], dtype=np.float64)
    class_labels_desc = [CONTIGUOUS_TO_NAME[c] for c in sorted_desc_indices]
    uniform_mean_train = np.mean(train_vals_desc)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    x_pos = np.arange(len(sorted_desc_indices))
    ax1.bar(x_pos, train_vals_desc, color="#2b5c8f", alpha=0.88, width=0.8, label="Immagini per classe (train2017)")
    ax1.axhline(
        uniform_mean_train,
        color="#d62728",
        linestyle="--",
        linewidth=1.8,
        label=f"Uniformità ideale (media = {uniform_mean_train:.0f} img/classe)",
    )
    ax1.set_yscale("log")
    ax1.set_title("Distribuzione delle 80 Classi Thing in COCO train2017 (Scala Log)", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Classi ordinate dalla più frequente alla più rara (1 → 80)", fontsize=10)
    ax1.set_ylabel("Numero di immagini (scala logaritmica)", fontsize=10)
    ax1.grid(True, which="both", linestyle=":", alpha=0.5)

    tick_step = 5
    ax1.set_xticks(x_pos[::tick_step])
    ax1.set_xticklabels([class_labels_desc[i] for i in range(0, len(class_labels_desc), tick_step)], rotation=45, ha="right", fontsize=8)
    ax1.annotate(
        f"Max: {class_labels_desc[0]} ({int(train_vals_desc[0])})",
        xy=(0, train_vals_desc[0]),
        xytext=(6, train_vals_desc[0] * 0.72),
        arrowprops=dict(arrowstyle="->", color="black", lw=1),
        fontsize=9,
        fontweight="bold",
    )
    ax1.annotate(
        f"Min: {class_labels_desc[-1]} ({int(train_vals_desc[-1])})",
        xy=(79, train_vals_desc[-1]),
        xytext=(52, train_vals_desc[-1] * 1.45),
        arrowprops=dict(arrowstyle="->", color="black", lw=1),
        fontsize=9,
        fontweight="bold",
    )
    ax1.legend(loc="upper right", fontsize=9)

    train_asc = np.sort(np.array([train_counts[c] for c in all_c_indices], dtype=np.float64))
    val_asc = np.sort(np.array([val_counts[c] for c in all_c_indices], dtype=np.float64))
    lx_train, ly_train, gini_train = compute_lorenz_and_gini(train_asc)
    lx_val, ly_val, gini_val = compute_lorenz_and_gini(val_asc)

    ax2.plot([0, 1], [0, 1], color="#d62728", linestyle="--", linewidth=1.8, label="Uniformità Perfetta (Gini = 0.000)")
    ax2.plot(lx_train, ly_train, color="#2b5c8f", linewidth=2.2, label=f"COCO train2017 intero (Gini = {gini_train:.3f})")
    ax2.plot(lx_val, ly_val, color="#ff7f0e", linestyle="-.", linewidth=1.8, label=f"COCO val2017 intero (Gini = {gini_val:.3f})")

    if balanced_subset_counts is not None:
        sub_asc = np.sort(np.array([balanced_subset_counts[c] for c in all_c_indices], dtype=np.float64))
        lx_sub, ly_sub, gini_sub = compute_lorenz_and_gini(sub_asc)
        ax2.plot(
            lx_sub,
            ly_sub,
            color="#2ca02c",
            linewidth=2.2,
            label=f"Subset Bilanciato N={subset_size} (Gini = {gini_sub:.3f})",
        )

    ax2.set_title("Curva di Lorenz — Misura di Disuguaglianza tra Classi", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Quota cumulata delle 80 classi (dalla più rara alla più frequente)", fontsize=10)
    ax2.set_ylabel("Quota cumulata delle occorrenze nel dataset", fontsize=10)
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.grid(True, linestyle=":", alpha=0.6)
    ax2.legend(loc="upper left", fontsize=9)

    save_and_show_fig(fig, "02_step2_coco_uniformity_lorenz_gini.png", save_svg=True)


def plot_detailed_class_comparison(
    train_counts: Counter,
    total_train_imgs: int,
    subset_counts: Counter,
    total_subset_imgs: int,
) -> None:
    """Genera e salva il Cleveland Dumbbell Chart per le 80 classi COCO Thing."""
    all_c_indices = list(range(1, NUM_CLASSES))
    sorted_desc = sorted(all_c_indices, key=lambda c: train_counts[c], reverse=True)
    labels = [f"{idx+1:02d}. {CONTIGUOUS_TO_NAME[c].title()}" for idx, c in enumerate(sorted_desc)]

    full_raw = np.array([train_counts[c] for c in sorted_desc], dtype=np.float64)
    sub_raw = np.array([subset_counts[c] for c in sorted_desc], dtype=np.float64)
    full_pct = (full_raw / float(total_train_imgs)) * 100.0
    sub_pct = (sub_raw / float(total_subset_imgs)) * 100.0
    boost_ratio = sub_pct / np.maximum(full_pct, 1e-9)

    y_pos = np.arange(len(sorted_desc))
    fig, (ax_left, ax_right) = plt.subplots(
        1, 2, figsize=(16, 22), gridspec_kw={"width_ratios": [3.0, 1.2]}, sharey=True
    )

    for idx in range(0, len(sorted_desc), 2):
        ax_left.axhspan(idx - 0.5, idx + 0.5, color="#f4f6f8", zorder=0)
        ax_right.axhspan(idx - 0.5, idx + 0.5, color="#f4f6f8", zorder=0)

    for idx in range(len(sorted_desc)):
        seg_color = "#2ca02c" if sub_pct[idx] >= full_pct[idx] else "#e67e22"
        ax_left.plot([full_pct[idx], sub_pct[idx]], [y_pos[idx], y_pos[idx]], color=seg_color, linewidth=2.2, alpha=0.85, zorder=2)

    ax_left.scatter(full_pct, y_pos, color="#1f4e78", s=48, zorder=3, label=f"Dataset Completo train2017 (N = {total_train_imgs:,})")
    ax_left.scatter(sub_pct, y_pos, color="#2ca02c", s=52, zorder=4, label=f"Subset Bilanciato Greedy (N = {total_subset_imgs:,})")

    min_quota_pct = (1.0 / len(all_c_indices)) * 100.0
    ax_left.axvline(min_quota_pct, color="#d62728", linestyle="--", linewidth=1.6, zorder=1, label=f"Quota minima garantita subset ({min_quota_pct:.2f}%)")

    ax_left.set_xscale("log")
    ax_left.set_xlim(0.10, 115.0)
    ax_left.set_xticks([0.1, 0.25, 0.5, 1.0, 2.5, 5.0, 10.0, 25.0, 50.0, 100.0])
    ax_left.set_xticklabels(["0.1%", "0.25%", "0.5%", "1%", "2.5%", "5%", "10%", "25%", "50%", "100%"], fontsize=9.5)

    for idx in range(len(sorted_desc)):
        left_val = min(full_pct[idx], sub_pct[idx])
        right_val = max(full_pct[idx], sub_pct[idx])
        ax_left.text(left_val * 0.88, y_pos[idx], f"{full_pct[idx]:.2f}%" if full_pct[idx] < 1.0 else f"{full_pct[idx]:.1f}%", va="center", ha="right", fontsize=7.8, color="#1f4e78")
        ax_left.text(right_val * 1.12, y_pos[idx], f"{sub_pct[idx]:.2f}%" if sub_pct[idx] < 1.0 else f"{sub_pct[idx]:.1f}%", va="center", ha="left", fontsize=7.8, fontweight="bold", color="#1b661b" if sub_pct[idx] >= full_pct[idx] else "#9c5211")

    ax_left.set_yticks(y_pos)
    ax_left.set_yticklabels(labels, fontsize=9.5)
    ax_left.invert_yaxis()
    ax_left.set_xlabel("Prevalenza nel Split (% immagini, scala logaritmica)", fontsize=11, fontweight="bold")
    ax_left.set_title("Confronto Prevalenza delle 80 Classi: Dataset Completo vs Subset Bilanciato", fontsize=13, fontweight="bold", pad=12)
    ax_left.grid(axis="x", which="major", linestyle=":", alpha=0.6)
    ax_left.legend(loc="lower right", fontsize=10, frameon=True, shadow=True)

    ratio_colors = ["#2ca02c" if r >= 1.0 else "#e67e22" for r in boost_ratio]
    ax_right.barh(y_pos, boost_ratio, height=0.62, color=ratio_colors, alpha=0.88, zorder=2)
    ax_right.axvline(1.0, color="black", linestyle="--", linewidth=1.3, zorder=3, label="Invariato (1.0x)")
    ax_right.set_xlim(0, boost_ratio.max() * 1.22)

    for idx, r_val in enumerate(boost_ratio):
        ax_right.text(r_val + 0.12, y_pos[idx], f"{r_val:.2f}x", va="center", ha="left", fontsize=8, fontweight="bold" if r_val >= 1.5 else "normal", color="#1b661b" if r_val >= 1.0 else "#9c5211")

    ax_right.set_xlabel("Moltiplicatore (Subset % / Full %)", fontsize=11, fontweight="bold")
    ax_right.set_title("Ribilanciamento", fontsize=13, fontweight="bold", pad=12)
    ax_right.grid(axis="x", linestyle=":", alpha=0.6)
    ax_right.legend(loc="lower right", fontsize=9.5, frameon=True)

    for ax in (ax_left, ax_right):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    save_and_show_fig(fig, "03_step2_coco_class_distribution_dumbbell.png", save_svg=True)


class COCOSemanticDataset(Dataset):
    """
    Dataset PyTorch per Semantic Segmentation su COCO 2017 (Thing).
    Implementa rasterizzazione istanze -> maschera 2D (0=bg, 1..80=classi) e
    Opzione C (resize proporzionale + RandomCrop/hflip in train, CenterCrop in val).
    """

    def __init__(
        self,
        coco_api: COCO,
        img_dir: str,
        img_ids: list,
        crop_size: int = CROP_SIZE,
        is_train: bool = True,
        cache_in_ram: bool = False,
    ):
        self.coco = coco_api
        self.img_dir = img_dir
        self.img_ids = list(img_ids)
        self.crop_size = int(crop_size)
        self.is_train = is_train
        self.cache_in_ram = cache_in_ram
        self._ram_cache = {}

        if self.cache_in_ram:
            print(f"Precaricamento in RAM di {len(self.img_ids)} coppie (is_train={self.is_train})...")
            t_start = time.perf_counter()
            for idx in range(len(self.img_ids)):
                self._ram_cache[idx] = self._load_raw_pair(idx)
            print(f"Precaricamento completato in {time.perf_counter() - t_start:.1f}s.")

    def __len__(self) -> int:
        return len(self.img_ids)

    def _load_raw_pair(self, idx: int):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, img_info["file_name"])

        with Image.open(img_path) as im:
            img_pil = im.convert("RGB")

        h, w = img_info["height"], img_info["width"]
        mask_np = np.zeros((h, w), dtype=np.uint8)

        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
        anns_sorted = sorted(anns, key=lambda a: a["area"], reverse=True)
        for ann in anns_sorted:
            c_idx = COCO_ID_TO_CONTIGUOUS[ann["category_id"]]
            binary_mask = self.coco.annToMask(ann)
            mask_np[binary_mask == 1] = c_idx

        mask_pil = Image.fromarray(mask_np, mode="L")
        return img_pil, mask_pil

    def _apply_synchronized_transforms(self, img_pil: Image.Image, mask_pil: Image.Image):
        img_resized = TF.resize(
            img_pil,
            size=self.crop_size,
            interpolation=InterpolationMode.BILINEAR,
            antialias=True,
        )
        mask_resized = TF.resize(
            mask_pil,
            size=self.crop_size,
            interpolation=InterpolationMode.NEAREST,
        )

        if self.is_train:
            i, j, h, w = transforms.RandomCrop.get_params(
                img_resized, output_size=(self.crop_size, self.crop_size)
            )
            img_cropped = TF.crop(img_resized, i, j, h, w)
            mask_cropped = TF.crop(mask_resized, i, j, h, w)

            # Usa torch.rand per mantenere l'intera stocasticità governata dal Generator PyTorch
            if torch.rand(1).item() < 0.5:
                img_cropped = TF.hflip(img_cropped)
                mask_cropped = TF.hflip(mask_cropped)
        else:
            img_cropped = TF.center_crop(img_resized, output_size=(self.crop_size, self.crop_size))
            mask_cropped = TF.center_crop(mask_resized, output_size=(self.crop_size, self.crop_size))

        img_tensor = TF.to_tensor(img_cropped)
        img_tensor = TF.normalize(img_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
        mask_tensor = torch.from_numpy(np.array(mask_cropped, dtype=np.int64))
        return img_tensor, mask_tensor

    def __getitem__(self, idx: int):
        if self.cache_in_ram:
            img_pil, mask_pil = self._ram_cache[idx]
        else:
            img_pil, mask_pil = self._load_raw_pair(idx)
        return self._apply_synchronized_transforms(img_pil, mask_pil)


if RUN_EDA_PLOTS:
    train_class_dist = analyze_coco_split(coco_train, "TRAIN 2017", target_short_side=CROP_SIZE)
    val_class_dist = analyze_coco_split(coco_val, "VAL 2017", target_short_side=CROP_SIZE)
    preview_subset_ids = select_class_balanced_subset(coco_train, max_samples=12000, seed=SEED)
    preview_subset_counts = compute_class_counts_from_ids(coco_train, preview_subset_ids)

    plot_dataset_uniformity(
        train_counts=train_class_dist,
        val_counts=val_class_dist,
        balanced_subset_counts=preview_subset_counts,
        subset_size=len(preview_subset_ids),
    )
    plot_detailed_class_comparison(
        train_counts=train_class_dist,
        total_train_imgs=len(coco_train.getImgIds()),
        subset_counts=preview_subset_counts,
        total_subset_imgs=len(preview_subset_ids),
    )

4. **Cella 4 — Modello FCN, Calibrazione Hardware e Istanziazione Unica dei DataLoader (Sezioni 8–9)**

In [ ]:
# ==============================================================================
# SEZIONE 8 & 9: MODELLO FCN, CALIBRAZIONE GPU E ISTANZIAZIONE DATALOADER
# (BATCH_SIZE viene calibrato PRIMA di creare i DataLoader: zero disallineamenti)
# ==============================================================================
def build_fcn_from_scratch(num_classes: int = NUM_CLASSES, aux_loss: bool = True) -> nn.Module:
    """Istanzia FCN-ResNet50 completamente da zero (weights=None, weights_backbone=None)."""
    return fcn_resnet50(
        weights=None,
        weights_backbone=None,
        num_classes=num_classes,
        aux_loss=aux_loss,
    )


def calibrate_batch_size_and_device(num_classes: int = NUM_CLASSES, crop_size: int = CROP_SIZE):
    """Calibra il BATCH_SIZE ottimale sulla GPU attiva prima di istanziare i DataLoader."""
    if not torch.cuda.is_available():
        print("ATTENZIONE: GPU non rilevata. Imposto BATCH_SIZE = 4 per esecuzione su CPU.")
        return 4, False

    num_gpus = torch.cuda.device_count()
    use_dp = num_gpus > 1
    print(f"=== CALIBRAZIONE GPU E BATCH SIZE ({num_gpus} GPU rilevate | DataParallel={use_dp}) ===")

    best_bs = DEFAULT_BATCH_SIZE
    for candidate_bs in [8, 16, 32]:
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            test_model = build_fcn_from_scratch(num_classes=num_classes, aux_loss=True).to(DEVICE)
            if use_dp:
                test_model = nn.DataParallel(test_model, device_ids=list(range(num_gpus)))
            test_model.train()

            dummy_x = torch.randn(candidate_bs, 3, crop_size, crop_size, device=DEVICE)
            dummy_y = torch.randint(0, num_classes, (candidate_bs, crop_size, crop_size), device=DEVICE, dtype=torch.long)
            criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
            optimizer = torch.optim.SGD(test_model.parameters(), lr=1e-3)

            t0 = time.perf_counter()
            optimizer.zero_grad(set_to_none=True)
            out = test_model(dummy_x)
            loss = criterion(out["out"], dummy_y) + AUX_LOSS_WEIGHT * criterion(out["aux"], dummy_y)
            loss.backward()
            optimizer.step()
            torch.cuda.synchronize()
            step_ms = (time.perf_counter() - t0) * 1000.0

            peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
            print(f"  - BATCH_SIZE = {candidate_bs:2d} -> OK | VRAM picco: {peak_vram_gb:.2f} GB | Tempo step: {step_ms:.1f} ms")
            best_bs = candidate_bs

            del test_model, dummy_x, dummy_y, out, loss, optimizer
            torch.cuda.empty_cache()
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"  - BATCH_SIZE = {candidate_bs:2d} -> OOM (Out of Memory)")
                torch.cuda.empty_cache()
                break
            raise e

    chosen_bs = min(best_bs, 16) if best_bs >= 16 else best_bs
    print(f"--> BATCH_SIZE operativo selezionato: {chosen_bs}\n")
    return chosen_bs, use_dp


BATCH_SIZE, USE_DATA_PARALLEL = calibrate_batch_size_and_device(num_classes=NUM_CLASSES, crop_size=CROP_SIZE)

# Generazione e salvataggio su disco dei pesi iniziali condivisi (per equità sperimentale Step 4)
seed_everything(SEED)
_init_ref_model = build_fcn_from_scratch(num_classes=NUM_CLASSES, aux_loss=True)
SHARED_INIT_WEIGHTS = copy.deepcopy(_init_ref_model.state_dict())
SHARED_INIT_PATH = os.path.join(CHECKPOINTS_DIR, "shared_init_fcn_resnet50.pt")
torch.save(SHARED_INIT_WEIGHTS, SHARED_INIT_PATH)

total_params = sum(p.numel() for p in _init_ref_model.parameters())
trainable_params = sum(p.numel() for p in _init_ref_model.parameters() if p.requires_grad)
del _init_ref_model

print(f"=== RIEPILOGO ARCHITETTURA E PESI INIZIALI ===")
print(f"Parametri totali / addestrabili : {total_params:,} / {trainable_params:,}")
print(f"Checkpoint pesi iniziali salvato: {SHARED_INIT_PATH}")

# Istanziazione UNICA dei Dataset e DataLoader con il BATCH_SIZE calibrato
train_ids = select_class_balanced_subset(coco_train, max_samples=TRAIN_MAX_SAMPLES, seed=SEED)
val_ids = sorted(coco_val.getImgIds())

train_dataset = COCOSemanticDataset(
    coco_api=coco_train,
    img_dir=os.path.join(DATA_ROOT, "train2017"),
    img_ids=train_ids,
    crop_size=CROP_SIZE,
    is_train=True,
    cache_in_ram=CACHE_IN_RAM,
)

val_dataset = COCOSemanticDataset(
    coco_api=coco_val,
    img_dir=os.path.join(DATA_ROOT, "val2017"),
    img_ids=val_ids,
    crop_size=CROP_SIZE,
    is_train=False,
    cache_in_ram=False,
)

g_train = torch.Generator()
g_train.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    generator=g_train,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    drop_last=False,
)

print(f"\n=== RIEPILOGO DATALOADER OPERATIVI ===")
print(f"Train samples: {len(train_dataset)} | Train batches: {len(train_loader)} (batch_size={train_loader.batch_size})")
print(f"Val samples  : {len(val_dataset)} | Val batches  : {len(val_loader)} (batch_size={val_loader.batch_size})")

# Sanity check veloce su 1 batch
t_batch = time.perf_counter()
images_batch, masks_batch = next(iter(train_loader))
print(f"Sanity 1° batch ({time.perf_counter() - t_batch:.3f}s) -> "
      f"images: {tuple(images_batch.shape)} [{images_batch.min():.2f}, {images_batch.max():.2f}] | "
      f"masks: {tuple(masks_batch.shape)} [{masks_batch.min().item()}, {masks_batch.max().item()}]")

if RUN_EDA_PLOTS:
    num_vis = min(4, BATCH_SIZE)
    fig, axes = plt.subplots(num_vis, 3, figsize=(12, 3.5 * num_vis))
    if num_vis == 1:
        axes = np.expand_dims(axes, axis=0)

    for idx in range(num_vis):
        img_vis = denormalize_image(images_batch[idx])
        mask_vis = masks_batch[idx].cpu().numpy()
        mask_rgb = PALETTE[mask_vis]
        overlay = (0.55 * img_vis + 0.45 * (mask_rgb / 255.0)).clip(0, 1)

        present_ids = [c for c in np.unique(mask_vis) if c != 0]
        labels_str = ", ".join(CONTIGUOUS_TO_NAME[c] for c in present_ids) if present_ids else "solo background"

        axes[idx, 0].imshow(img_vis)
        axes[idx, 0].set_title(f"Immagine ({CROP_SIZE}x{CROP_SIZE})")
        axes[idx, 1].imshow(mask_rgb)
        axes[idx, 1].set_title(f"Maschera ({labels_str})")
        axes[idx, 2].imshow(overlay)
        axes[idx, 2].set_title("Overlay Allineamento")
        for col in range(3):
            axes[idx, col].axis("off")

    save_and_show_fig(fig, "01_step2_dataloader_sanity_overlay.png")

5. **Cella 5 — Funzioni Diagnostiche Incapsulate: Babysitting & Mini-Ablation (Sezioni 10, 11 e 11-BIS)**

In [ ]:
# ==============================================================================
# SEZIONI 10, 11 & 11-BIS: DIAGNOSTICA BABYSITTING E MINI-ABLATION (INCAPSULATE)
# (Eseguite SOLO se RUN_BABYSITTING = True; saltate istantaneamente per Step 4+)
# ==============================================================================
def compute_update_to_weight_ratios(model: nn.Module, lr: float) -> dict:
    """Calcola il rapporto ||lr * grad||_2 / ||weight||_2 (CS231n) su layer rappresentativi."""
    target_layers = {
        "backbone.conv1 (input)": model.backbone.conv1.weight,
        "backbone.layer4 (deep)": model.backbone.layer4[-1].conv3.weight,
        "classifier.4 (head)": model.classifier[4].weight,
    }
    ratios = {}
    for name, param in target_layers.items():
        if param.grad is not None:
            w_norm = param.data.norm(2).item()
            upd_norm = (lr * param.grad.data).norm(2).item()
            ratios[name] = upd_norm / max(w_norm, 1e-12)
    return ratios


def compute_batch_miou_and_acc(logits: torch.Tensor, targets: torch.Tensor, num_classes: int = NUM_CLASSES):
    """Termometro rapido batch-wise per il solo test di overfitting dello Step 3."""
    preds = torch.argmax(logits, dim=1)
    valid = targets != IGNORE_INDEX
    correct = ((preds == targets) & valid).sum().item()
    total = valid.sum().item()
    pix_acc = correct / max(1, total)

    ious = []
    for c in range(num_classes):
        pred_c = (preds == c) & valid
        targ_c = (targets == c) & valid
        union = (pred_c | targ_c).sum().item()
        if union > 0:
            inter = (pred_c & targ_c).sum().item()
            ious.append(inter / union)
    mean_iou = float(np.mean(ious)) if ious else 0.0
    return pix_acc, mean_iou


def run_step3_babysitting(babysit_ids: list, num_epochs: int = 80, lr: float = 1e-3) -> None:
    """Esegue Sanity Check Step 0, rapporto Update/Weight e test di overfitting su 16 immagini."""
    seed_everything(SEED)
    babysit_dataset = COCOSemanticDataset(
        coco_api=coco_train,
        img_dir=os.path.join(DATA_ROOT, "train2017"),
        img_ids=babysit_ids,
        crop_size=CROP_SIZE,
        is_train=False,
        cache_in_ram=True,
    )
    babysit_loader = DataLoader(babysit_dataset, batch_size=8, shuffle=False)
    criterion_ce = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    # Sanity Check 1: Loss teorica vs empirica a pesi casuali (Step 0)
    fcn_sanity = build_fcn_from_scratch(num_classes=NUM_CLASSES, aux_loss=True).to(DEVICE)
    fcn_sanity.train()
    bx, by = next(iter(babysit_loader))
    bx, by = bx.to(DEVICE), by.to(DEVICE)
    with torch.no_grad():
        out_init = fcn_sanity(bx)
        loss_main_init = criterion_ce(out_init["out"], by).item()
        loss_aux_init = criterion_ce(out_init["aux"], by).item()
        loss_tot_init = loss_main_init + AUX_LOSS_WEIGHT * loss_aux_init

    expected_ce = float(np.log(NUM_CLASSES))
    print(f"\n=== SANITY CHECK 1: LOSS INIZIALE (PESI CASUALI, STEP 0) ===")
    print(f"Loss teorica attesa -ln(1/{NUM_CLASSES}) : {expected_ce:.4f} (totale con aux: {(1.0 + AUX_LOSS_WEIGHT) * expected_ce:.4f})")
    print(f"Loss empirica misurata (main)   : {loss_main_init:.4f} (totale con aux: {loss_tot_init:.4f})")
    del fcn_sanity

    # Sanity Check 2 & 3: Overfitting controllato su 16 immagini
    seed_everything(SEED)
    fcn_overfit = build_fcn_from_scratch(num_classes=NUM_CLASSES, aux_loss=True).to(DEVICE)
    if USE_DATA_PARALLEL:
        fcn_overfit = nn.DataParallel(fcn_overfit, device_ids=list(range(torch.cuda.device_count())))

    optimizer_overfit = torch.optim.AdamW(fcn_overfit.parameters(), lr=lr, weight_decay=0.0)
    history_babysit = {"epoch": [], "loss": [], "pix_acc": [], "miou": []}

    print(f"\n=== SANITY CHECK 2 & 3: OVERFITTING SU 16 IMMAGINI ({num_epochs} EPOCHE) ===")
    t_ov_start = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        fcn_overfit.train()
        epoch_loss, epoch_acc, epoch_miou, n_batches = 0.0, 0.0, 0.0, 0

        for imgs_b, masks_b in babysit_loader:
            imgs_b = imgs_b.to(DEVICE, non_blocking=True)
            masks_b = masks_b.to(DEVICE, non_blocking=True)

            optimizer_overfit.zero_grad(set_to_none=True)
            outputs = fcn_overfit(imgs_b)
            loss = criterion_ce(outputs["out"], masks_b) + AUX_LOSS_WEIGHT * criterion_ce(outputs["aux"], masks_b)
            loss.backward()

            if epoch == 1 and n_batches == 0:
                base_model = fcn_overfit.module if isinstance(fcn_overfit, nn.DataParallel) else fcn_overfit
                ratios = compute_update_to_weight_ratios(base_model, lr=lr)
                print("Rapporto Update/Weight al Step 1 (target CS231n ~ 1e-3):")
                for lname, rval in ratios.items():
                    print(f"  - {lname:25s}: {rval:.2e}")

            optimizer_overfit.step()

            with torch.no_grad():
                acc_b, miou_b = compute_batch_miou_and_acc(outputs["out"], masks_b, NUM_CLASSES)

            epoch_loss += loss.item()
            epoch_acc += acc_b
            epoch_miou += miou_b
            n_batches += 1

        history_babysit["epoch"].append(epoch)
        history_babysit["loss"].append(epoch_loss / n_batches)
        history_babysit["pix_acc"].append(epoch_acc / n_batches)
        history_babysit["miou"].append(epoch_miou / n_batches)

        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch [{epoch:02d}/{num_epochs:02d}] | Loss: {history_babysit['loss'][-1]:.4f} | "
                  f"Pixel Acc: {history_babysit['pix_acc'][-1]*100:.2f}% | mIoU (presenti): {history_babysit['miou'][-1]*100:.2f}%")

    print(f"Test di overfitting completato in {time.perf_counter() - t_ov_start:.1f}s.")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    ax1.plot(history_babysit["epoch"], history_babysit["loss"], color="#d62728", linewidth=2)
    ax1.set_title("Babysitting: Curva Training Loss (16 immagini)", fontweight="bold")
    ax1.set_xlabel("Epoca")
    ax1.set_ylabel("Cross-Entropy Loss (Main + 0.5 Aux)")
    ax1.grid(True, linestyle=":", alpha=0.6)

    ax2.plot(history_babysit["epoch"], [a * 100 for a in history_babysit["pix_acc"]], label="Pixel Accuracy (%)", color="#1f4e78", linewidth=2)
    ax2.plot(history_babysit["epoch"], [m * 100 for m in history_babysit["miou"]], label="mIoU Classi Presenti (%)", color="#2ca02c", linewidth=2)
    ax2.set_title("Babysitting: Convergenza Accuracy e mIoU", fontweight="bold")
    ax2.set_xlabel("Epoca")
    ax2.set_ylabel("Percentuale (%)")
    ax2.set_ylim(0, 105)
    ax2.grid(True, linestyle=":", alpha=0.6)
    ax2.legend()
    save_and_show_fig(fig, "04_step3_babysitting_loss_metrics_curves.png", save_svg=True)

    fcn_overfit.eval()
    vis_x, vis_y = next(iter(babysit_loader))
    with torch.no_grad():
        vis_preds = torch.argmax(fcn_overfit(vis_x.to(DEVICE))["out"], dim=1).cpu().numpy()

    fig, axes = plt.subplots(4, 3, figsize=(12, 13))
    for idx in range(4):
        axes[idx, 0].imshow(denormalize_image(vis_x[idx]))
        axes[idx, 0].set_title(f"Input ({CROP_SIZE}x{CROP_SIZE})")
        axes[idx, 1].imshow(PALETTE[vis_y[idx].numpy()])
        axes[idx, 1].set_title("Ground Truth")
        axes[idx, 2].imshow(PALETTE[vis_preds[idx]])
        axes[idx, 2].set_title("Predizione FCN (Overfit)")
        for c in range(3):
            axes[idx, c].axis("off")

    save_and_show_fig(fig, "05_step3_babysitting_overfit_predictions.png")

    del fcn_overfit, optimizer_overfit, babysit_dataset, babysit_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _evaluate_ablation_subset(model: nn.Module, loader: DataLoader, num_classes: int = NUM_CLASSES) -> dict:
    """Valutazione deterministica per la mini-ablation dello Step 3-BIS."""
    model.eval()
    inter_total = np.zeros(num_classes, dtype=np.float64)
    union_total = np.zeros(num_classes, dtype=np.float64)
    gt_present = np.zeros(num_classes, dtype=bool)
    correct_pix, valid_pix = 0, 0

    with torch.no_grad():
        for imgs_b, masks_b in loader:
            imgs_b = imgs_b.to(DEVICE)
            masks_np = masks_b.numpy()
            preds_np = torch.argmax(model(imgs_b)["out"], dim=1).cpu().numpy()

            valid = masks_np != IGNORE_INDEX
            correct_pix += np.sum((preds_np == masks_np) & valid)
            valid_pix += np.sum(valid)

            for c in range(num_classes):
                p_c = (preds_np == c) & valid
                t_c = (masks_np == c) & valid
                if np.any(t_c):
                    gt_present[c] = True
                inter_total[c] += np.sum(p_c & t_c)
                union_total[c] += np.sum(p_c | t_c)

    ious = np.zeros(num_classes, dtype=np.float64)
    for c in range(num_classes):
        if union_total[c] > 0:
            ious[c] = inter_total[c] / union_total[c]

    present_all = np.where(gt_present)[0]
    present_fg = np.where(gt_present & (np.arange(num_classes) > 0))[0]
    return {
        "pix_acc": correct_pix / max(1, valid_pix),
        "miou_all": float(np.mean(ious[present_all])) if len(present_all) > 0 else 0.0,
        "miou_fg": float(np.mean(ious[present_fg])) if len(present_fg) > 0 else 0.0,
        "detected_fg": int(np.sum(ious[present_fg] > 0.10)),
        "total_fg": len(present_fg),
    }


def run_step3bis_mini_ablation(babysit_ids: list) -> None:
    """Esegue il confronto controllato T1 (256px) vs T2 (CosineLR) vs T3 (384px)."""
    trials_config = [
        ("T1: Baseline (256px, 80 ep, LR fisso)", 256, 80, False),
        ("T2: Scheduler (256px, 150 ep, CosineLR)", 256, 150, True),
        ("T3: Risoluzione (384px, 150 ep, CosineLR)", 384, 150, True),
    ]
    ablation_results, ablation_visuals = {}, {}

    print("\n=== ESECUZIONE MINI-ABLATION DIAGNOSTICA (STEP 3-BIS) ===")
    for t_name, c_sz, n_ep, use_sch in trials_config:
        seed_everything(SEED)
        ds = COCOSemanticDataset(
            coco_api=coco_train,
            img_dir=os.path.join(DATA_ROOT, "train2017"),
            img_ids=babysit_ids,
            crop_size=c_sz,
            is_train=False,
            cache_in_ram=True,
        )
        loader = DataLoader(ds, batch_size=8, shuffle=False)

        model = build_fcn_from_scratch(num_classes=NUM_CLASSES, aux_loss=True)
        model.load_state_dict(copy.deepcopy(SHARED_INIT_WEIGHTS))
        model = model.to(DEVICE)
        if USE_DATA_PARALLEL:
            model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))

        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.0)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_ep, eta_min=1e-5) if use_sch else None
        criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

        t0 = time.perf_counter()
        final_loss = 0.0
        for _ in range(n_ep):
            model.train()
            ep_loss, n_b = 0.0, 0
            for imgs_b, masks_b in loader:
                imgs_b = imgs_b.to(DEVICE, non_blocking=True)
                masks_b = masks_b.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                out = model(imgs_b)
                loss = criterion(out["out"], masks_b) + AUX_LOSS_WEIGHT * criterion(out["aux"], masks_b)
                loss.backward()
                optimizer.step()
                ep_loss += loss.item()
                n_b += 1
            if scheduler is not None:
                scheduler.step()
            final_loss = ep_loss / n_b

        elapsed_s = time.perf_counter() - t0
        metrics = _evaluate_ablation_subset(model, loader, NUM_CLASSES)
        metrics["final_loss"] = final_loss
        metrics["time_s"] = elapsed_s
        ablation_results[t_name] = metrics

        model.eval()
        vis_x, vis_y = next(iter(loader))
        with torch.no_grad():
            vis_preds = torch.argmax(model(vis_x.to(DEVICE))["out"], dim=1).cpu().numpy()

        ablation_visuals[t_name] = {
            "imgs": [denormalize_image(vis_x[i]) for i in range(4)],
            "gts": [vis_y[i].numpy() for i in range(4)],
            "preds": [vis_preds[i] for i in range(4)],
        }

        del model, optimizer, scheduler, ds, loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\n" + "=" * 102)
    print(f"{'Configurazione Tentativo':42s} | {'Loss':>7s} | {'PixAcc':>7s} | {'mIoU All':>8s} | {'mIoU Obj':>8s} | {'Obj Rilevati':>12s} | {'Tempo':>6s}")
    print("-" * 102)
    for t_name, res in ablation_results.items():
        print(
            f"{t_name:42s} | {res['final_loss']:7.4f} | {res['pix_acc']*100:6.2f}% | "
            f"{res['miou_all']*100:7.2f}% | {res['miou_fg']*100:7.2f}% | "
            f"{res['detected_fg']:5d}/{res['total_fg']:<6d} | {res['time_s']:5.1f}s"
        )
    print("=" * 102)

    t_names = list(ablation_visuals.keys())
    fig, axes = plt.subplots(4, 5, figsize=(19, 14))
    col_titles = ["Input (384px)", "Ground Truth", "T1: Baseline 256px", "T2: CosineLR 256px", "T3: CosineLR 384px"]
    for row in range(4):
        axes[row, 0].imshow(ablation_visuals[t_names[2]]["imgs"][row])
        axes[row, 1].imshow(PALETTE[ablation_visuals[t_names[2]]["gts"][row]])
        axes[row, 2].imshow(PALETTE[ablation_visuals[t_names[0]]["preds"][row]])
        axes[row, 3].imshow(PALETTE[ablation_visuals[t_names[1]]["preds"][row]])
        axes[row, 4].imshow(PALETTE[ablation_visuals[t_names[2]]["preds"][row]])
        for col in range(5):
            if row == 0:
                axes[row, col].set_title(col_titles[col], fontsize=10, fontweight="bold")
            axes[row, col].axis("off")

    save_and_show_fig(fig, "06_step3bis_mini_ablation_visual_comparison.png")


if RUN_BABYSITTING:
    babysit_ids = select_class_balanced_subset(coco_train, max_samples=100, seed=SEED)[:16]
    run_step3_babysitting(babysit_ids, num_epochs=80, lr=1e-3)
    run_step3bis_mini_ablation(babysit_ids)
else:
    print("[Info] RUN_BABYSITTING = False -> Test diagnostici Step 3 e 3-BIS saltati. Ambiente pronto per lo Step 4.")